# Etapa 4 — Perfiles, validación estadística y outliers (pipeline principal)

Clustering jerárquico **Aglomerativo Ward** en dos niveles:

- **Nivel 1 (K=4)**: separa 3 grupos especiales + mainstream.
- **Nivel 2**: subdivisión del mainstream.

Para cada nivel: listado de hospitales por cluster, perfiles, pruebas estadísticas por tipo de variable (Kruskal-Wallis numéricas / Chi-cuadrado categóricas) con ranking de discriminación, y descripción de cada cluster **derivada del análisis** (no se asignan nombres a priori). Cierra con detección de outliers y la comparación externa vs MINSAL.

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert np.__version__.startswith('1.'), 'NumPy debe ser 1.x'

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.stats import kruskal, chi2_contingency
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, silhouette_samples)
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest

from src.utils.io import read_parquet, TABLES_DIR, FIGURES_DIR, PROCESSED_DIR

ALPHA = 0.05
RANDOM_STATE = 42
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 60)

## 4.1 Carga de matriz, asignaciones (nivel 1, K=4) y nombres

In [2]:
state = json.loads((PROCESSED_DIR / 'pipeline_state.json').read_text())
K_OPTIMO = state['K_optimo']
matriz = read_parquet('hospital_matrix')
matriz['COD_HOSPITAL'] = matriz['COD_HOSPITAL'].astype(str)
asign = read_parquet('hospital_clusters_integrado')
asign['COD_HOSPITAL'] = asign['COD_HOSPITAL'].astype(str)

# Nombres de hospitales (tabla maestra)
maestra = pd.read_excel(ROOT / 'insumos' / 'maestras' / 'Tablas maestras bases GRD.xlsx', sheet_name='Hospitales',
    header=None, skiprows=1, names=['COD_HOSPITAL','NOMBRE'])
maestra['COD_HOSPITAL'] = maestra['COD_HOSPITAL'].astype(str).str.strip()
nombres = dict(zip(maestra['COD_HOSPITAL'], maestra['NOMBRE']))

df = matriz.merge(asign, on='COD_HOSPITAL', how='inner')
df['NOMBRE'] = df['COD_HOSPITAL'].map(nombres)
print(f'Nivel 1 — K={K_OPTIMO} | distribución:', df['cluster'].value_counts().sort_index().to_dict())

Nivel 1 — K=4 | distribución: {0: 3, 1: 53, 2: 2, 3: 7}


## 4.2 Hospitales en cada cluster (nivel 1)

In [3]:
lst = df[['cluster','COD_HOSPITAL','NOMBRE']].sort_values(['cluster','COD_HOSPITAL'])
lst.to_csv(TABLES_DIR / 'hospitales_por_cluster_nivel1.csv', index=False)
for c in sorted(df['cluster'].unique()):
    sub = lst[lst['cluster']==c]
    print(f"\n=== Cluster C{c} (n={len(sub)}) ===")
    for _, r in sub.iterrows():
        print(f"  {r['COD_HOSPITAL']}  {str(r['NOMBRE'])[:60]}")


=== Cluster C0 (n=3) ===
  109101  Hospital Clínico de Niños Dr. Roberto del Río (Santiago, Ind
  112102  Hospital de Niños Dr. Luis Calvo Mackenna (Santiago, Provide
  113130  Hospital Dr. Exequiel González Cortés (Santiago, San Miguel)

=== Cluster C1 (n=53) ===
  101100  Hospital Dr. Juan Noé Crevanni (Arica)
  102100  Hospital Dr. Ernesto Torres Galdames (Iquique)
  103100  Hospital Dr. Leonardo Guzmán (Antofagasta)
  103101  Hospital Dr. Carlos Cisternas (Calama)
  104100  Hospital San José del Carmen (Copiapó)
  104103  Hospital Provincial del Huasco Monseñor Fernando Ariztía Rui
  105100  Hospital San Juan de Dios (La Serena)
  105101  Hospital San Pablo (Coquimbo)
  105102  Hospital Dr. Antonio Tirado Lanas (Ovalle)
  106100  Hospital Carlos Van Buren (Valparaíso)
  106103  Hospital Claudio Vicuña (San Antonio)
  107100  Hospital Dr. Gustavo Fricke (Viña del Mar)
  107101  Hospital San Martín (Quillota)
  107102  Hospital de Quilpué
  108100  Hospital de San Camilo (San Felipe

## 4.3 Perfil de cada cluster (media de todas las variables)

In [4]:
cols_perfil = [c for c in matriz.columns
    if c not in ('COD_HOSPITAL','peso_medio_cma_imputado') and not c.startswith('dim_')]
perfiles = df.groupby('cluster')[cols_perfil].mean()
perfiles.insert(0, 'n', df.groupby('cluster').size())
perfiles.to_csv(TABLES_DIR / 'perfiles_clusters_integrado.csv')
print(perfiles.T.to_string(float_format='%.3f'))

cluster                        0         1        2        3
n                          3.000    53.000    2.000    7.000
egresos_por_anio        7200.944 14048.129 3681.667 5735.024
estancia_media             6.021     6.638    7.876    8.447
estancia_mediana           2.667     3.236    3.500    4.571
peso_medio_grd             1.224     0.964    2.003    1.087
severidad_media            1.907     1.677    1.857    1.859
mortalidad_media           1.699     1.504    1.665    1.731
entropia_grd               0.778     0.816    0.738    0.833
comorbilidades_promedio    2.420     2.831    3.317    3.182
tasa_cma                   0.142     0.148    0.000    0.220
peso_medio_cma             0.420     0.541    0.695    0.589
cv_estancia                2.566     1.966    2.338    1.899
edad_mediana               4.969    41.914   56.916   58.058
pabellones_promedio        0.499     0.533    1.192    0.366
pct_alta_domicilio         0.956     0.879    0.920    0.803
pct_alta_fallecido      

## 4.4 Pruebas estadísticas por variable (nivel 1)

**Kruskal-Wallis** para variables numéricas (categoría = cluster). p≤0.05 ⇒ la variable discrimina los grupos. Se ordenan por el estadístico H (mayor H = discrimina más fuerte).

In [5]:
num_cols = [c for c in matriz.columns
    if c not in ('COD_HOSPITAL','peso_medio_cma_imputado') and not c.startswith('dim_')
    and pd.api.types.is_numeric_dtype(matriz[c])]
grupos = sorted(df['cluster'].unique())
filas = []
for col in num_cols:
    muestras = [df.loc[df['cluster']==g, col].dropna().values for g in grupos]
    muestras = [m for m in muestras if len(m) > 0]
    try:
        H, p = kruskal(*muestras)
    except ValueError:
        H, p = np.nan, np.nan
    fila = {'variable': col, 'H': H, 'p_valor': p,
            'discrimina': 'si' if (pd.notna(p) and p<=ALPHA) else 'no'}
    for g in grupos:
        fila[f'med_C{g}'] = df.loc[df['cluster']==g, col].median()
    filas.append(fila)
kw = pd.DataFrame(filas).sort_values('H', ascending=False).reset_index(drop=True)
kw.to_csv(TABLES_DIR / 'pruebas_kruskal_nivel1.csv', index=False)
n_sig = (kw['discrimina']=='si').sum()
print(f'Variables que discriminan (p<=0.05): {n_sig}/{len(kw)}\n')
print('TOP variables que MÁS discriminan (mayor H):')
print(kw[['variable','H','p_valor','discrimina']].head(12).to_string(index=False, float_format='%.4g'))
print('\nVariables que NO discriminan:')
print(kw[kw['discrimina']=='no'][['variable','H','p_valor']].to_string(index=False, float_format='%.4g'))

Variables que discriminan (p<=0.05): 24/28

TOP variables que MÁS discriminan (mayor H):
               variable     H   p_valor discrimina
            tasa_partos 29.01 2.231e-06         si
           edad_mediana  28.3 3.134e-06         si
 pct_obstetrica_ingreso 25.68 1.112e-05         si
         pct_pediatrico 25.34  1.31e-05         si
         pct_geriatrico 24.24 2.229e-05         si
    pct_femenino_fertil 23.21 3.645e-05         si
     pct_alta_domicilio 22.11 6.191e-05         si
        severidad_media 20.68 0.0001227         si
       mortalidad_media 20.68 0.0001227         si
        tasa_prematurez 20.63 0.0001256         si
       pct_uso_pabellon 19.86 0.0001813         si
comorbilidades_promedio 18.12 0.0004163         si

Variables que NO discriminan:
             variable     H  p_valor
   pct_estancia_larga 7.513  0.05723
pct_origen_emergencia 7.039  0.07067
             tasa_cma 6.103   0.1067
          cv_estancia 5.577   0.1341


## 4.5 Chi-cuadrado: variable categórica (Servicio de Salud) vs cluster

In [6]:
g = read_parquet('grd_filtrado')[['COD_HOSPITAL','SERVICIO_SALUD']]
g['COD_HOSPITAL'] = g['COD_HOSPITAL'].astype(str)
serv = g.drop_duplicates('COD_HOSPITAL').set_index('COD_HOSPITAL')['SERVICIO_SALUD']
df['SERVICIO_SALUD'] = df['COD_HOSPITAL'].map(serv)
tabla = pd.crosstab(df['cluster'], df['SERVICIO_SALUD'])
chi2, p_chi, dof, _ = chi2_contingency(tabla)
pd.DataFrame([{'variable':'SERVICIO_SALUD','chi2':chi2,'gl':dof,'p_valor':p_chi,
    'discrimina':'si' if p_chi<=ALPHA else 'no'}]).to_csv(
    TABLES_DIR / 'pruebas_chi2_nivel1.csv', index=False)
print(f'Chi2={chi2:.2f} gl={dof} p={p_chi:.3f} -> '
      f"{'discrimina' if p_chi<=ALPHA else 'NO discrimina (clusters no son geográficos)'}")

Chi2=60.54 gl=84 p=0.975 -> NO discrimina (clusters no son geográficos)


## 4.6 Descripción de cada cluster (derivada del análisis)

Cada cluster se describe **a partir de las variables que más lo distinguen** (mediana del cluster vs mediana global), sin imponer una etiqueta clínica a priori.

In [7]:
vars_sig = kw[kw['discrimina']=='si']['variable'].tolist()
glob_med = df[vars_sig].median()
def describir(cluster_id):
    sub = df[df['cluster']==cluster_id]
    difs = []
    for v in vars_sig:
        cm, gm = sub[v].median(), glob_med[v]
        if gm != 0:
            ratio = cm / gm
        else:
            ratio = np.inf if cm > 0 else 1.0
        difs.append((v, cm, gm, ratio))
    # ordenar por desvío relativo respecto a la mediana global
    difs.sort(key=lambda x: abs(np.log((x[3]+1e-9))), reverse=True)
    return difs[:6]

filas_desc = []
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster']==c]
    print(f"\n=== Cluster C{c} (n={len(sub)}) — rasgos que lo distinguen ===")
    for v, cm, gm, ratio in describir(c):
        signo = 'ALTO' if ratio > 1.15 else ('BAJO' if ratio < 0.85 else '~medio')
        print(f'  {v:<24} cluster={cm:.3f}  global={gm:.3f}  ({signo}, x{ratio:.2f})')
        filas_desc.append({'cluster':f'C{c}','variable':v,'mediana_cluster':cm,
            'mediana_global':gm,'ratio':ratio,'nivel':signo})
pd.DataFrame(filas_desc).to_csv(TABLES_DIR / 'descripcion_clusters_nivel1.csv', index=False)


=== Cluster C0 (n=3) — rasgos que lo distinguen ===
  tasa_partos              cluster=0.000  global=0.027  (BAJO, x0.00)
  pct_obstetrica_ingreso   cluster=0.000  global=0.193  (BAJO, x0.00)
  pct_geriatrico           cluster=0.000  global=0.257  (BAJO, x0.00)
  tasa_prematurez          cluster=0.000  global=0.065  (BAJO, x0.00)
  pct_femenino_fertil      cluster=0.026  global=0.317  (BAJO, x0.08)
  edad_mediana             cluster=4.999  global=41.993  (BAJO, x0.12)

=== Cluster C1 (n=53) — rasgos que lo distinguen ===
  egresos_por_anio         cluster=13136.833  global=10666.833  (ALTO, x1.23)
  tasa_prematurez          cluster=0.078  global=0.065  (ALTO, x1.20)
  tasa_partos              cluster=0.028  global=0.027  (~medio, x1.05)
  pct_programada           cluster=0.173  global=0.179  (~medio, x0.97)
  pct_obstetrica_ingreso   cluster=0.200  global=0.193  (~medio, x1.03)
  pct_femenino_fertil      cluster=0.325  global=0.317  (~medio, x1.03)

=== Cluster C2 (n=2) — rasgos que l

## 4.7 Subclustering del mainstream (nivel 2)

El cluster mayoritario (mainstream) se subdivide con Aglomerativo Ward. Se elige K_sub buscando subgrupos representativos (n≥4). El Silhouette interno bajo confirma que el mainstream es un continuo: la subdivisión se justifica por las diferencias entre variables (Kruskal nivel 2), no por separación geométrica.

In [8]:
COLS_DROP = ['COD_HOSPITAL','peso_medio_cma','peso_medio_cma_imputado','cluster','NOMBRE','SERVICIO_SALUD']
feats = df.drop(columns=[c for c in COLS_DROP if c in df.columns]).copy()
feats.index = df['COD_HOSPITAL'].values
feats = feats.fillna(0.0)
for c in ['egresos_por_anio','edad_mediana','pabellones_promedio']:
    if c in feats.columns: feats[c] = np.log1p(feats[c].clip(lower=0))
X = RobustScaler().fit_transform(feats.values)
ids = feats.index.tolist()
labels1 = df.set_index('COD_HOSPITAL').loc[ids, 'cluster'].values

from collections import Counter
main_c = Counter(labels1).most_common(1)[0][0]
mask_main = labels1 == main_c
X_main = X[mask_main]; ids_main = [ids[i] for i in range(len(ids)) if mask_main[i]]
Zm = linkage(X_main, method='ward')
rows = []
for ks in range(2, 9):
    lab = fcluster(Zm, t=ks, criterion='maxclust') - 1
    sizes = sorted(pd.Series(lab).value_counts().tolist(), reverse=True)
    sil = silhouette_score(X_main, lab) if len(set(lab))>1 else np.nan
    n4 = sum(1 for s in sizes if s>=4)
    rows.append({'K_sub':ks,'silhouette':sil,'n_subclusters_n4plus':n4,
        'n_singletons':sum(1 for s in sizes if s==1),'tamanos':str(sizes)})
df_sub = pd.DataFrame(rows)
df_sub.to_csv(TABLES_DIR / 'subclustering_mainstream_metricas.csv', index=False)
print(f'Mainstream = C{main_c} (n={int(mask_main.sum())})')
print(df_sub.to_string(index=False, float_format='%.3f'))
# Elegir K_sub: maximizar subclusters n>=4 penalizando singletons
df_sub['score'] = df_sub['n_subclusters_n4plus']*10 - df_sub['n_singletons']*3
K_SUB = int(df_sub.loc[df_sub['score'].idxmax(), 'K_sub'])
print(f'\n>>> K_sub elegido (max subclusters n>=4): {K_SUB}')

Mainstream = C1 (n=53)
 K_sub  silhouette  n_subclusters_n4plus  n_singletons                    tamanos
     2       0.200                     2             0                   [39, 14]
     3       0.073                     3             0               [23, 16, 14]
     4       0.084                     3             1            [23, 16, 13, 1]
     5       0.109                     4             1          [23, 16, 9, 4, 1]
     6       0.113                     5             1      [16, 13, 10, 9, 4, 1]
     7       0.124                     6             1   [16, 13, 10, 5, 4, 4, 1]
     8       0.093                     7             1 [16, 10, 7, 6, 5, 4, 4, 1]

>>> K_sub elegido (max subclusters n>=4): 8


### 4.7b Hospitales en cada subcluster del mainstream

In [9]:
lab_sub = fcluster(Zm, t=K_SUB, criterion='maxclust') - 1
sub_map = dict(zip(ids_main, lab_sub))
asig_jer = []
for cod in ids:
    c1 = df.set_index('COD_HOSPITAL').loc[cod,'cluster']
    if c1 == main_c:
        n1, n2 = 'MAINSTREAM', f'S{sub_map[cod]}'
    else:
        n1 = n2 = f'C{c1}'
    asig_jer.append({'COD_HOSPITAL':cod,'NOMBRE':nombres.get(cod,'?'),'nivel1':n1,'nivel2':n2})
df_jer = pd.DataFrame(asig_jer)
df_jer.to_csv(TABLES_DIR / 'asignacion_jerarquica.csv', index=False)
print('Distribución nivel2:', df_jer['nivel2'].value_counts().sort_index().to_dict())
for s in sorted(df_jer[df_jer['nivel1']=='MAINSTREAM']['nivel2'].unique()):
    sub = df_jer[df_jer['nivel2']==s]
    print(f"\n--- Subcluster {s} (n={len(sub)}) ---")
    for _, r in sub.iterrows():
        print(f"  {r['COD_HOSPITAL']}  {str(r['NOMBRE'])[:55]}")

Distribución nivel2: {'C0': 3, 'C2': 2, 'C3': 7, 'S0': 4, 'S1': 5, 'S2': 4, 'S3': 1, 'S4': 16, 'S5': 10, 'S6': 6, 'S7': 7}

--- Subcluster S0 (n=4) ---
  107102  Hospital de Quilpué
  115110  Hospital de Santa Cruz
  121121  Hospital de Villarrica
  128109  Hospital Provincial Dr. Rafael Avaría (Curanilahue)

--- Subcluster S1 (n=5) ---
  103101  Hospital Dr. Carlos Cisternas (Calama)
  106103  Hospital Claudio Vicuña (San Antonio)
  113150  Hospital San Luis (Buin)
  113180  Hospital El Pino (Santiago, San Bernardo)
  118105  Hospital San José (Coronel)

--- Subcluster S2 (n=4) ---
  109100  Complejo Hospitalario San José (Santiago, Independencia
  110130  Hospital Adalberto Steeger (Talagante)
  110150  Hospital San José (Melipilla)
  112101  Hospital Dr. Luis Tisné B. (Santiago, Peñalolén)

--- Subcluster S3 (n=1) ---
  111100  Hospital Clínico San Borja-Arriarán (Santiago, Santiago

--- Subcluster S4 (n=16) ---
  101100  Hospital Dr. Juan Noé Crevanni (Arica)
  102100  Hospital Dr.

## 4.8 Pruebas estadísticas dentro del mainstream (nivel 2)

Kruskal-Wallis entre los subclusters del mainstream: identifica qué variables separan los subgrupos del núcleo (donde el Silhouette ya no ayuda).

In [10]:
dfm = df[df['cluster']==main_c].copy()
dfm['sub'] = dfm['COD_HOSPITAL'].map(sub_map)
sub_grupos = sorted(dfm['sub'].unique())
filas2 = []
for col in num_cols:
    muestras = [dfm.loc[dfm['sub']==g, col].dropna().values for g in sub_grupos]
    muestras = [m for m in muestras if len(m) > 0]
    try:
        H, p = kruskal(*muestras)
    except ValueError:
        H, p = np.nan, np.nan
    filas2.append({'variable':col,'H':H,'p_valor':p,
        'discrimina':'si' if (pd.notna(p) and p<=ALPHA) else 'no'})
kw2 = pd.DataFrame(filas2).sort_values('H', ascending=False).reset_index(drop=True)
kw2.to_csv(TABLES_DIR / 'pruebas_kruskal_nivel2_mainstream.csv', index=False)
print(f'Variables que discriminan en el mainstream (p<=0.05): '
      f"{(kw2['discrimina']=='si').sum()}/{len(kw2)}\n")
print('TOP variables que más separan los subgrupos del mainstream:')
print(kw2[['variable','H','p_valor','discrimina']].head(10).to_string(index=False, float_format='%.4g'))

Variables que discriminan en el mainstream (p<=0.05): 24/28

TOP variables que más separan los subgrupos del mainstream:
              variable     H   p_valor discrimina
        peso_medio_grd 40.41 1.052e-06         si
        pct_programada 39.08 1.886e-06         si
pct_obstetrica_ingreso 34.75 1.248e-05         si
          pct_urgencia 33.06 2.579e-05         si
       tasa_prematurez 32.15 3.817e-05         si
    pct_estancia_larga 31.67 4.675e-05         si
          entropia_grd 31.46 5.119e-05         si
 pct_origen_emergencia 29.85 0.0001012         si
   pct_femenino_fertil  29.8 0.0001031         si
      mortalidad_media 29.57  0.000114         si


## 4.9 Comparación externa vs clasificación de complejidad MINSAL

In [11]:
BASE = ROOT / 'info-hospitales' / 'Base de Establecimientos 2023.xlsx'
base = pd.read_excel(BASE, skiprows=1)
base.columns = [str(c).strip() for c in base.columns]
base['COD_HOSPITAL'] = base['Código Vigente'].astype(str).str.replace('.0','',regex=False).str.strip()
minsal = (base.dropna(subset=['Nivel de Complejidad']).drop_duplicates('COD_HOSPITAL')
              .set_index('COD_HOSPITAL')['Nivel de Complejidad'])
dcomp = df[['COD_HOSPITAL','cluster']].copy()
dcomp['minsal'] = dcomp['COD_HOSPITAL'].map(minsal)
dcomp = dcomp.dropna(subset=['minsal'])
cont = pd.crosstab(dcomp['cluster'], dcomp['minsal'])
cont.to_csv(TABLES_DIR / 'comparacion_minsal_contingencia.csv')
ari = adjusted_rand_score(dcomp['minsal'], dcomp['cluster'])
nmi = normalized_mutual_info_score(dcomp['minsal'], dcomp['cluster'])
pd.DataFrame([{'comparacion':f'Ward K={K_OPTIMO} vs MINSAL','n':len(dcomp),'ARI':ari,'NMI':nmi}]).to_csv(
    TABLES_DIR / 'comparacion_minsal_metricas.csv', index=False)
print('Contingencia cluster x MINSAL:'); print(cont.to_string())
print(f'\nARI={ari:.3f} | NMI={nmi:.3f} (n={len(dcomp)})')

Contingencia cluster x MINSAL:
minsal   Alta Complejidad  Mediana Complejidad
cluster                                       
0                       3                    0
1                      47                    6
2                       2                    0
3                       3                    4

ARI=0.177 | NMI=0.125 (n=65)


## 4.10 Detección de outliers (hospitales atípicos)

Tres criterios complementarios sobre la matriz escalada (35 features):

1. **Silhouette negativo**: hospital peor ubicado en su cluster que en el vecino.
2. **Isolation Forest**: detector multivariado de anomalías (contaminación 10%).
3. **Distancia al centroide**: z-score > 3 respecto a la distancia media de su cluster.

Un hospital marcado por ≥2 criterios se considera atípico robusto.

In [12]:
# 1) Silhouette por hospital (en la partición K=4)
sil_samples = silhouette_samples(X, labels1)
out = pd.DataFrame({'COD_HOSPITAL': ids, 'NOMBRE': [nombres.get(i,'?') for i in ids],
    'cluster': labels1, 'silhouette': sil_samples})
out['sil_negativo'] = out['silhouette'] < 0

# 2) Isolation Forest
iso = IsolationForest(contamination=0.10, random_state=RANDOM_STATE, n_estimators=300)
out['iso_outlier'] = iso.fit_predict(X) == -1
out['iso_score'] = iso.score_samples(X)

# 3) Distancia al centroide del cluster (z-score intra-cluster)
dist_cent = np.zeros(len(ids))
for c in np.unique(labels1):
    m = labels1 == c
    cent = X[m].mean(axis=0)
    d = np.linalg.norm(X[m] - cent, axis=1)
    dist_cent[m] = d
out['dist_centroide'] = dist_cent
z = np.zeros(len(ids))
for c in np.unique(labels1):
    m = labels1 == c
    mu, sd = dist_cent[m].mean(), dist_cent[m].std(ddof=0)
    z[m] = (dist_cent[m] - mu) / sd if sd > 0 else 0.0
out['z_dist'] = z
out['dist_outlier'] = out['z_dist'] > 3

out['n_criterios'] = out[['sil_negativo','iso_outlier','dist_outlier']].sum(axis=1)
out['outlier_robusto'] = out['n_criterios'] >= 2
out['outlier_union'] = out['n_criterios'] >= 1
out = out.sort_values(['n_criterios','iso_score'], ascending=[False, True])
out.to_csv(TABLES_DIR / 'outliers_hospitales.csv', index=False)

print('Outliers por Isolation Forest:', int(out['iso_outlier'].sum()))
print('Outliers por Silhouette negativo:', int(out['sil_negativo'].sum()))
print('Outliers por distancia (z>3):', int(out['dist_outlier'].sum()))
rob = out[out['outlier_robusto']]
uni = out[out['outlier_union']]
print(f'\nAtípicos por >=2 criterios (robustos): {len(rob)}')
print(f'Atípicos por >=1 criterio (unión): {len(uni)}')
print('\nHospitales atípicos (unión de criterios):')
for _, r in uni.iterrows():
    crit = []
    if r['sil_negativo']: crit.append('sil<0')
    if r['iso_outlier']: crit.append('isoForest')
    if r['dist_outlier']: crit.append('dist>3sd')
    print(f"  C{int(r['cluster'])}  {r['COD_HOSPITAL']}  {str(r['NOMBRE'])[:45]:<45} [{', '.join(crit)}]")
print('\nTop 8 más atípicos (Isolation Forest):')
print(out.head(8)[['COD_HOSPITAL','NOMBRE','cluster','silhouette','iso_score','z_dist']].to_string(index=False, float_format='%.3f'))

Outliers por Isolation Forest: 7
Outliers por Silhouette negativo: 1
Outliers por distancia (z>3): 1

Atípicos por >=2 criterios (robustos): 0
Atípicos por >=1 criterio (unión): 9

Hospitales atípicos (unión de criterios):
  C2  112103  Instituto Nacional de Enfermedades Respirator [isoForest]
  C2  112104  Instituto de Neurocirugía Dr. Alfonso Asenjo  [isoForest]
  C3  106102  Hospital Dr. Eduardo Pereira Ramírez (Valpara [isoForest]
  C0  113130  Hospital Dr. Exequiel González Cortés (Santia [isoForest]
  C3  111195  Hospital de Urgencia Asistencia Pública Dr. A [isoForest]
  C0  112102  Hospital de Niños Dr. Luis Calvo Mackenna (Sa [isoForest]
  C3  112100  Hospital Del Salvador (Santiago, Providencia) [isoForest]
  C1  111100  Hospital Clínico San Borja-Arriarán (Santiago [dist>3sd]
  C3  116110  Hospital San José (Parral)                    [sil<0]

Top 8 más atípicos (Isolation Forest):
COD_HOSPITAL                                                              NOMBRE  cluster  sil

### 4.10b Figura de outliers (PCA 2D)

In [13]:
from sklearn.decomposition import PCA
coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
out_idx = {cod:i for i,cod in enumerate(ids)}
fig, ax = plt.subplots(figsize=(11,7))
ax.scatter(coords[:,0], coords[:,1], c=labels1, cmap='tab10', s=60,
           edgecolors='black', linewidth=0.5, alpha=0.8)
marca = out[out['outlier_union']]
for _, r in marca.iterrows():
    i = out_idx[r['COD_HOSPITAL']]
    ax.scatter(coords[i,0], coords[i,1], s=240, facecolors='none',
               edgecolors='red', linewidth=2.2, zorder=5)
    ax.annotate(str(r['NOMBRE'])[:22], (coords[i,0], coords[i,1]),
                fontsize=7, xytext=(5,5), textcoords='offset points')
ax.set_title('Outliers (círculo rojo = atípico por ≥1 criterio) — Ward K=4')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'outliers_pca2d.png', dpi=150, bbox_inches='tight'); plt.close()
print('Figura: outliers_pca2d.png')

Figura: outliers_pca2d.png


## 4.11 Generar RESUMEN_PIPELINE_INTEGRADO.md

In [14]:
from datetime import datetime
kw_no = kw[kw['discrimina']=='no']['variable'].tolist()
top_disc = kw.head(6)['variable'].tolist()
L = []
L.append('# Resumen Pipeline Integrado (principal, sin camas)\n')
L.append(f'_Generado: {datetime.now():%Y-%m-%d %H:%M}_\n')
L.append('## Configuración\n')
L.append(f'- Técnica: **Aglomerativo Ward** jerárquico de 2 niveles')
L.append(f'- Hospitales: {state["n_hospitales"]} | Features: {state["n_features"]} (sin camas)')
L.append(f'- Nivel 1: K={K_OPTIMO} | Nivel 2: subdivisión del mainstream en K_sub={K_SUB}')
L.append(f'- K natural (corte trivial): {state["K_natural"]} (Sil {state["silhouette_K_natural"]:.3f})\n')
L.append('## Distribución de clusters\n')
L.append(f'- Nivel 1: {df["cluster"].value_counts().sort_index().to_dict()}')
L.append(f'- Nivel 2: {df_jer["nivel2"].value_counts().sort_index().to_dict()}\n')
L.append('## Pruebas estadísticas\n')
L.append(f'- Nivel 1 (Kruskal): {(kw["discrimina"]=="si").sum()}/{len(kw)} variables discriminan')
L.append(f'- Top discriminantes: {top_disc}')
L.append(f'- No discriminan: {kw_no}')
L.append(f'- Nivel 2 mainstream (Kruskal): {(kw2["discrimina"]=="si").sum()}/{len(kw2)} variables discriminan')
L.append(f'- Chi² Servicio de Salud: p={p_chi:.3f} (no discrimina: clusters no son geográficos)\n')
L.append('## Comparación MINSAL\n')
L.append(f'- ARI={ari:.3f} | NMI={nmi:.3f} (n={len(dcomp)})\n')
L.append('## Outliers\n')
L.append(f'- Isolation Forest: {int(out["iso_outlier"].sum())} | Silhouette<0: {int(out["sil_negativo"].sum())} | dist>3sd: {int(out["dist_outlier"].sum())}')
L.append(f'- Atípicos (>=1 criterio): {uni["COD_HOSPITAL"].tolist()}')
L.append(f'- Atípicos robustos (>=2 criterios): {rob["COD_HOSPITAL"].tolist() or "ninguno (criterios capturan facetas distintas)"}')
(ROOT / 'reports' / 'RESUMEN_PIPELINE_INTEGRADO.md').write_text('\n'.join(L), encoding='utf-8')
print('RESUMEN_PIPELINE_INTEGRADO.md generado')
print('\n'.join(L))

RESUMEN_PIPELINE_INTEGRADO.md generado
# Resumen Pipeline Integrado (principal, sin camas)

_Generado: 2026-05-30 23:49_

## Configuración

- Técnica: **Aglomerativo Ward** jerárquico de 2 niveles
- Hospitales: 65 | Features: 35 (sin camas)
- Nivel 1: K=4 | Nivel 2: subdivisión del mainstream en K_sub=8
- K natural (corte trivial): 2 (Sil 0.663)

## Distribución de clusters

- Nivel 1: {0: 3, 1: 53, 2: 2, 3: 7}
- Nivel 2: {'C0': 3, 'C2': 2, 'C3': 7, 'S0': 4, 'S1': 5, 'S2': 4, 'S3': 1, 'S4': 16, 'S5': 10, 'S6': 6, 'S7': 7}

## Pruebas estadísticas

- Nivel 1 (Kruskal): 24/28 variables discriminan
- Top discriminantes: ['tasa_partos', 'edad_mediana', 'pct_obstetrica_ingreso', 'pct_pediatrico', 'pct_geriatrico', 'pct_femenino_fertil']
- No discriminan: ['pct_estancia_larga', 'pct_origen_emergencia', 'tasa_cma', 'cv_estancia']
- Nivel 2 mainstream (Kruskal): 24/28 variables discriminan
- Chi² Servicio de Salud: p=0.975 (no discrimina: clusters no son geográficos)

## Comparación MINSAL

- 